In [1]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [2]:
DATA_DIR = Path("../data/interim/ieee-fraud-detection")
TARGET_COLUMN = "isFraud"
ID_COLUMN = "TransactionID"
TIME_COLUMN = "TransactionDT"

time_col = TIME_COLUMN
target_col = TARGET_COLUMN
train_ratio = 0.8
train_search_width = 0.05
min_test_pos = 200

In [ ]:
df = pd.read_parquet(DATA_DIR / "test.parquet")

,TransactionDT,TransactionAmt,card1,card2,card3,card5,addr1,addr2,dist1,dist2,...,id_17,id_18,id_19,id_20,id_21,id_22,id_24,id_25,id_26,id_32
count,5.066910e+05,506691.000000,506691.000000,498037.000000,503689.000000,502144.000000,441082.000000,441082.000000,215474.000000,36436.000000,...,135966.000000,50875.000000,135906.000000,135633.000000,5059.000000,5062.000000,4740.000000,5039.000000,5047.000000,70671.000000
mean,2.692994e+07,134.725568,9957.222175,363.735379,153.543409,200.162975,291.846514,86.723412,87.065270,237.175047,...,191.070341,14.795735,350.122982,408.886230,507.727021,15.336823,13.166667,332.043064,152.752923,26.217939
std,4.756507e+06,245.779822,4884.960969,158.688653,12.443013,40.562461,102.062730,2.987328,314.131694,556.450834,...,30.749535,2.318496,139.140824,158.971756,227.371061,5.618032,3.222440,86.356683,31.916995,3.601046
min,1.840322e+07,0.018000,1001.000000,100.000000,100.000000,100.000000,100.000000,10.000000,0.000000,0.000000,...,100.000000,11.000000,100.000000,100.000000,100.000000,11.000000,10.000000,100.000000,100.000000,8.000000
25%,2.277154e+07,40.000000,6019.000000,207.000000,150.000000,166.000000,204.000000,87.000000,3.000000,7.000000,...,166.000000,13.000000,266.000000,256.000000,252.000000,14.000000,11.000000,321.000000,137.000000,24.000000
50%,2.720466e+07,67.950000,9803.000000,369.000000,150.000000,226.000000,299.000000,87.000000,8.000000,44.000000,...,166.000000,15.000000,321.000000,484.000000,576.000000,14.000000,11.000000,321.000000,147.000000,24.000000
75%,3.134856e+07,125.000000,14276.000000,512.000000,150.000000,226.000000,330.000000,87.000000,20.000000,196.000000,...,225.000000,15.000000,427.000000,549.000000,711.000000,14.000000,15.000000,355.000000,182.000000,32.000000
max,3.421434e+07,10270.000000,18397.000000,600.000000,232.000000,237.000000,540.000000,102.000000,8081.000000,9213.000000,...,228.000000,29.000000,670.000000,660.000000,854.000000,44.000000,26.000000,549.000000,216.000000,48.000000


In [6]:
idx = 1
df[df.columns[idx]].dtype, df.columns[idx]

(dtype('float64'), 'TransactionAmt')

In [16]:
df["id_01"].dtype

dtype('float64')

In [21]:
df[df["id_01"] % 1 != 0]["id_01"]

TransactionID
3663549   NaN
3663550   NaN
3663551   NaN
3663552   NaN
3663553   NaN
           ..
4170231   NaN
4170232   NaN
4170235   NaN
4170237   NaN
4170238   NaN
Name: id_01, Length: 364784, dtype: float64

In [ ]:
df = pd.read_parquet(DATA_DIR / "train.parquet")
df = df.sort_values(time_col)
n = len(df)
y = df[target_col].to_numpy()

In [4]:
train_range = range(
    max(1, int(n * (train_ratio - train_search_width))),
    min(n - 2, int(n * (train_ratio + train_search_width))) + 1
)

best = None
train_end = 0
best_score = float("inf")

for i in tqdm(train_range):
    y_train = y[: i]
    y_test = y[i: ]

    if len(y_test) == 0:
        continue

    n_test_pos = y_test.sum()

    if n_test_pos < min_test_pos:
        continue

    train_target_rate = y_train.mean()
    test_target_rate = y_test.mean()

    score = abs(train_target_rate - test_target_rate)

    if score < best_score:
        best_score = score
        train_end = i
        best = {
            "train_end": i,
            "train_rate": train_target_rate,
            "test_rate": test_target_rate,
            "train_pos": int(y_train.sum()),
            "test_pos": int(n_test_pos),
            "score": score,
        }

best

100%|██████████| 59055/59055 [00:32<00:00, 1820.86it/s]


{'train_end': 498700,
 'train_rate': np.float64(0.035001002606777624),
 'test_rate': np.float64(0.034930313588850175),
 'train_pos': 17455,
 'test_pos': 3208,
 'score': np.float64(7.068901792744997e-05)}

In [37]:
import datetime
df_main = df.iloc[: train_end]
START_DATE = datetime.datetime.strptime('2017-11-30', '%Y-%m-%d')
df_main['DT_M'] = df_main['TransactionDT'].apply(lambda x: (START_DATE + datetime.timedelta(seconds = x)))
df_main['DT_M'] = (df_main['DT_M'].dt.year - 2017) * 12 + df_main['DT_M'].dt.month 

/tmp/ipykernel_1820/1127569060.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_main['DT_M'] = df_main['TransactionDT'].apply(lambda x: (START_DATE + datetime.timedelta(seconds = x)))
/tmp/ipykernel_1820/1127569060.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_main['DT_M'] = (df_main['DT_M'].dt.year - 2017) * 12 + df_main['DT_M'].dt.month


In [38]:
df_main["DT_M"].value_counts()

DT_M
12    137321
15    101632
13     92585
14     86021
16     81141
Name: count, dtype: int64

In [39]:
df_main.groupby('DT_M')['isFraud'].sum()

DT_M
12    3550
13    3705
14    3447
15    4019
16    2734
Name: isFraud, dtype: int64